# Transform Payment Data
# Extrating Date and Time from payment timestamp and creating new Column for date and time
# Maping Payment to descriptive Values
# (1-Sucess,2-Pending,3-Cancelled,4-Failed)
# Writing Transform to Silver Schema

### Extracting Date and Time from Payment_timeStamp

In [0]:
df = spark.read.table('gizmobox_nara.bronze.payments')
display(df)

In [0]:
%sql
SELECT * From gizmobox_nara.bronze.payments

In [0]:
import pyspark.sql.functions as f
df_extracted_payment = (
                        df
                        .select(
                            "payment_Id",
                            "order_id",
                            f.date_format("payment_timestamp", "yyyy-MM-dd").alias("payment_date"),
                            f.date_format("payment_timestamp", "HH:mm:ss").alias("payment_time"),
                            "payment_status",
                            "payment_method"
                            )
)
display(df_extracted_payment)

In [0]:
%sql
SELECT payment_id,order_id,
date_format(payment_timestamp,'yyyy-MM-dd') AS payment_date,
date_format(payment_timestamp,'HH:mm:ss') AS payment_time,
payment_status,payment_method
FROM gizmobox_nara.bronze.payments


### Mapping Payment_Status to descriptive values

In [0]:
df_mapped_payment = (
                    df_extracted_payment
                    .select("order_id",
                            "payment_Id",
                            "payment_date",
                            "payment_time",
                            f.when(df_extracted_payment.payment_status == 1, "Success")
                            .when(df_extracted_payment.payment_status == 2, "Pending")
                            .when(df_extracted_payment.payment_status == 3, "cancelled")
                            .when(df_extracted_payment.payment_status == 4, "Refunded")
                            .alias("payment_status"),
                            "payment_method"
                            )
)
display(df_mapped_payment)


In [0]:
%sql
SELECT payment_id,order_id,
date_format(payment_timestamp,'yyyy-MM-dd') AS payment_date,
date_format(payment_timestamp,'HH:mm:ss') AS payment_time,
CASE payment_status
When 1 THEN 'Success'
When 2 THEN 'Pending'
When 3 THEN 'Cancelled'
When 4 Then 'Failed'
END AS payment_status,
payment_method
FROM gizmobox_nara.bronze.payments

## Write the Transformdata to Silver Schema

In [0]:
%sql
CREATE TABLE gizmobox_nara.silver.payments
SELECT payment_id,order_id,
date_format(payment_timestamp,'yyyy-MM-dd') AS payment_date,
date_format(payment_timestamp,'HH:mm:ss') AS payment_time,
CASE payment_status
When 1 THEN 'Success'
When 2 THEN 'Pending'
When 3 THEN 'Cancelled'
When 4 Then 'Failed'
END AS payment_status,
payment_method
FROM gizmobox_nara.bronze.payments

In [0]:
df_mapped_payment.writeTo("gizmobox_nara.silver.py_payment").createOrReplace()

In [0]:
%python
new = spark.read.table("gizmobox_nara.silver.py_payment")
display(new)

In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.payments